#### Etapa 1: Entendimento, EDA e Baselines

##### ML Canvas & Formulação do Problema
* **Problema de Negócio:** Operadora perdendo clientes. Precisamos classificar clientes com risco de cancelamento (Churn).
* **Métrica de Negócio:** Custo de churn evitado (focar no cliente certo economiza dinheiro).
* **Métrica Técnica:** **F1-Score** e **AUC-ROC**. A acurácia não serve bem aqui porque os dados são desbalanceados.
* **SLOs:** A API final de inferência deve ter baixa latência para integração real-time.

In [1]:
import sys
import os
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

sys.path.append(os.path.abspath(".."))
from src.utils.logger import get_logger

logger = get_logger("notebook_eda")

# carregando os dados
DATA_PATH = "../data/raw/telco_churn.csv"
DATASET_URL = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"

if not os.path.exists(DATA_PATH):
    logger.info("Baixando dataset...")
    df = pd.read_csv(DATASET_URL)
    df.to_csv(DATA_PATH, index=False)
else:
    logger.info("Dataset carregado do disco local.")
    df = pd.read_csv(DATA_PATH)

df.head(3)

2026-04-06 21:56:40 | INFO     | notebook_eda | Dataset carregado do disco local.


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes


In [2]:
logger.info("Limpando os dados...")

# 2- Remove ID (não ajuda na previsão)
df = df.drop(columns=['customerID'])

# 1- Arruma a coluna TotalCharges que veio como texto
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'].replace(' ', ''), errors='coerce').fillna(0)

# 3- Transforma 'Yes'/'No' em 1 e 0 (Binarização do Target)
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

logger.info(f"Dados limpos. Linhas e Colunas: {df.shape}")

# Porcentagem de desbalanceamento
display(df['Churn'].value_counts(normalize=True) * 100)

2026-04-06 21:56:53 | INFO     | notebook_eda | Limpando os dados...
2026-04-06 21:56:53 | INFO     | notebook_eda | Dados limpos. Linhas e Colunas: (7043, 20)


Churn
0    73.463013
1    26.536987
Name: proportion, dtype: float64